In [4]:
import torch
import torchaudio
import numpy as np
from pathlib import Path
import subprocess

# 设置设备
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# 加载 Silero VAD 模型
model, utils = torch.hub.load(repo_or_dir='snakers4/silero-vad', model='silero_vad', force_reload=False)
(get_speech_timestamps, save_audio, read_audio, VADIterator, collect_chunks) = utils
model = model.to(device)
print("Silero VAD model loaded")

Using device: cuda
Silero VAD model loaded


Using cache found in /home/cyijun/.cache/torch/hub/snakers4_silero-vad_master


In [5]:
def extract_audio(video_path, output_wav=None, sample_rate=16000):
    """从视频提取音频并转为16kHz单声道WAV"""
    video_path = Path(video_path)
    
    # 检查视频文件是否存在
    if not video_path.exists():
        raise FileNotFoundError(f"视频文件不存在: {video_path}")
    
    if output_wav is None:
        output_wav = video_path.with_suffix('.wav')
    else:
        output_wav = Path(output_wav)
    
    # 使用 ffmpeg 提取音频
    cmd = [
        'ffmpeg', '-y', '-i', str(video_path),
        '-vn',  # 无视频
        '-acodec', 'pcm_s16le',  # 16位PCM
        '-ac', '1',  # 单声道
        '-ar', str(sample_rate),  # 16kHz
        str(output_wav)
    ]
    
    # 运行 ffmpeg 并捕获输出以便调试
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"FFmpeg 错误码: {result.returncode}")
        print(f"FFmpeg 错误输出: {result.stderr}")
        raise subprocess.CalledProcessError(
            result.returncode, cmd, output=result.stdout, stderr=result.stderr
        )
    
    print(f"音频已提取到: {output_wav}")
    return output_wav

# 设置视频文件路径（请根据实际路径修改）
# Windows 路径示例: r"Z:\\new\\savr-401\\video.mp4"
# Linux/WSL 路径示例: "/mnt/z/new/savr-401/video.mp4"
video_file = "/mnt/mininas/4t2/new/savr-401/4k2.com@savr-401_1_8k.mp4"

# 检查文件是否存在
if not Path(video_file).exists():
    print(f"警告: 视频文件不存在: {video_file}")
    print("请修改 video_file 变量为正确的视频路径")

wav_file = extract_audio(video_file, output_wav=Path(video_file).with_suffix('.wav').name)
wav_file

音频已提取到: 4k2.com@savr-401_1_8k.wav


PosixPath('4k2.com@savr-401_1_8k.wav')

In [ ]:
wav_file='4k2.com@savr-401_1_8k.wav'

In [7]:
# 读取音频文件
wav = read_audio(str(wav_file), sampling_rate=16000).to(device)
print(f"Audio duration: {len(wav)/16000:.2f} seconds")

# 获取语音时间戳
# threshold: 语音检测阈值 (默认0.5)，越小越敏感
# min_speech_duration_ms: 最小语音段长度（毫秒）
# max_speech_duration_s: 最大语音段长度（秒）
# min_silence_duration_ms: 静音间隔超过此值则分段（毫秒）
speech_timestamps = get_speech_timestamps(
    wav, 
    model, 
    sampling_rate=16000,
    threshold=0.5,
    min_speech_duration_ms=250,      # 忽略小于250ms的语音
    max_speech_duration_s=float('inf'),
    min_silence_duration_ms=500,     # 500ms静音视为分段点
    window_size_samples=512
)

print(f"Found {len(speech_timestamps)} speech segments")
speech_timestamps[:5]  # 显示前5个

Audio duration: 1419.16 seconds
Found 311 speech segments


[{'start': 338464, 'end': 487904},
 {'start': 495648, 'end': 532448},
 {'start': 546848, 'end': 561632},
 {'start': 570400, 'end': 607200},
 {'start': 631840, 'end': 637920}]

In [9]:
import json

def save_segments(wav, timestamps, output_dir="segments", sampling_rate=16000):
    """将语音段保存为单独文件"""
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    segments_info = []
    
    for i, ts in enumerate(timestamps):
        start = ts['start'] / sampling_rate
        end = ts['end'] / sampling_rate
        duration = end - start
        
        # 提取音频段
        segment = wav[ts['start']:ts['end']]
        
        # 保存为WAV
        segment_file = output_dir / f"segment_{i:04d}_{start:.3f}_{end:.3f}.wav"
        torchaudio.save(str(segment_file), segment.unsqueeze(0), sampling_rate)
        
        segments_info.append({
            'index': i,
            'start': round(start, 3),
            'end': round(end, 3),
            'duration': round(duration, 3),
            'file': str(segment_file.name)
        })
    
    # 保存元数据
    with open(output_dir / 'segments.json', 'w', encoding='utf-8') as f:
        json.dump(segments_info, f, ensure_ascii=False, indent=2)
    
    print(f"Saved {len(timestamps)} segments to {output_dir}")
    return segments_info

# 保存语音段
segments = save_segments(wav.to('cpu'), speech_timestamps)
segments[:3]

Saved 311 segments to segments


[{'index': 0,
  'start': 21.154,
  'end': 30.494,
  'duration': 9.34,
  'file': 'segment_0000_21.154_30.494.wav'},
 {'index': 1,
  'start': 30.978,
  'end': 33.278,
  'duration': 2.3,
  'file': 'segment_0001_30.978_33.278.wav'},
 {'index': 2,
  'start': 34.178,
  'end': 35.102,
  'duration': 0.924,
  'file': 'segment_0002_34.178_35.102.wav'}]

In [10]:
# 显示分割结果摘要
total_duration = len(wav) / 16000
speech_duration = sum(s['end'] - s['start'] for s in speech_timestamps) / 16000
durations = [(s['end'] - s['start']) / 16000 for s in speech_timestamps]

print(f"=" * 50)
print(f"视频: {video_file}")
print(f"总时长: {total_duration:.2f}秒 ({total_duration/60:.2f}分钟)")
print(f"语音段数: {len(speech_timestamps)}")
print(f"语音总时长: {speech_duration:.2f}秒")
print(f"语音占比: {speech_duration/total_duration*100:.1f}%")
print(f"=" * 50)
print(f"统计信息:")
print(f"  平均每段: {speech_duration/len(speech_timestamps):.2f}秒")
print(f"  最短段:   {min(durations):.2f}秒")
print(f"  最长段:   {max(durations):.2f}秒")
print(f"=" * 50)
print("\n前10个语音段:")
for s in segments[:10]:
    print(f"  [{s['index']:03d}] {s['start']:>8.3f}s - {s['end']:>8.3f}s ({s['duration']:.2f}s)")

视频: /mnt/mininas/4t2/new/savr-401/4k2.com@savr-401_1_8k.mp4
总时长: 1419.16秒 (23.65分钟)
语音段数: 311
语音总时长: 859.20秒
语音占比: 60.5%
统计信息:
  平均每段: 2.76秒
  最短段:   0.32秒
  最长段:   15.04秒

前10个语音段:
  [000]   21.154s -   30.494s (9.34s)
  [001]   30.978s -   33.278s (2.30s)
  [002]   34.178s -   35.102s (0.92s)
  [003]   35.650s -   37.950s (2.30s)
  [004]   39.490s -   39.870s (0.38s)
  [005]   40.994s -   44.350s (3.36s)
  [006]   44.994s -   54.302s (9.31s)
  [007]   55.490s -   56.894s (1.40s)
  [008]   58.306s -   63.134s (4.83s)
  [009]   64.258s -   68.190s (3.93s)
